# GT |error| on the AF2 model of ubiquitin G76C (BMRB 27512_1)

Writes the |error| into the B-factor column (-1 = no data) and one `.pml`.

In [ ]:
import json
import os

import numpy as np
from Bio.PDB import PDBIO, PDBParser
from Bio.PDB.Polypeptide import three_to_index, index_to_one

PID = "27512_1"
PRED_PATH = ("../../../data/analysis/Fig5/predictions_N_ensemble_10seed_sorted_centered_err.json")
GT_PATH = ("../../../data/rBMRB/cluster_30p/PANAV/test.json")
AF2_DIR = ("../../../data/rBMRB/cluster_30p/structures_split_filtered/test")
OUT_DIR = os.getcwd()

SENTINEL = -1.00
VMIN, VMAX = 0.0, 5.0       # must match draw_colorbar / the other *_gt_error renders
TOP_THRESH = 5.0            # GT |error| above this: side-chain sticks + amide-N sphere

## 1. GT |error| per residue and sequence check

In [ ]:
pred_json = json.load(open(PRED_PATH))
gt_json = json.load(open(GT_PATH))
inv = {v: k for k, v in gt_json["entry_ID"].items()}
bmrb_seq = "".join(gt_json["sequence"][inv[PID]])

mask = np.array(pred_json["mask"][PID], dtype=bool)
oof_resid = np.array(pred_json["oof_resid"][PID], dtype=np.float64)
oof_mask = np.array(pred_json["oof_mask"][PID], dtype=bool)
n = min(len(mask), len(oof_resid), len(oof_mask), len(bmrb_seq))
valid = mask[:n] & oof_mask[:n] & np.isfinite(oof_resid[:n])
gt_err = {i: float(oof_resid[i]) for i in np.where(valid)[0]}      # 0-based BMRB index -> GT |error|


def load_chain(pid):
    struct = PDBParser(QUIET=True).get_structure(pid, os.path.join(AF2_DIR, f"{pid}.pdb"))
    chain = next(struct[0].get_chains())
    residues = [r for r in chain if r.id[0] == " "]
    return struct, residues


def one_letter(residues):
    return "".join(index_to_one(three_to_index(r.get_resname())) for r in residues)


af_struct, af_res = load_chain(PID)
af_seq = one_letter(af_res)
if af_seq != bmrb_seq:
    raise ValueError(f"AF2 sequence does not match BMRB sequence:\n AF2  {af_seq}\n BMRB {bmrb_seq}")
print(f"{PID}: {len(bmrb_seq)} residues, sequence matches AF2 model | GT |error| on {len(gt_err)} residues")

gt_n = gt_json["N"][inv[PID]]                                     # GT 15N shift (ppm), None = no data
order = sorted(gt_err, key=gt_err.get, reverse=True)
print(f"median {np.median(list(gt_err.values())):.2f} ppm, mean {np.mean(list(gt_err.values())):.2f} ppm")
print("largest |error| (GT 15N):", ", ".join(f"{bmrb_seq[i]}{i + 1} {gt_err[i]:.1f} (GT {gt_n[i]:.1f})" for i in order[:8]))
print(f"> {TOP_THRESH:g} ppm (shown as sticks):",
      ", ".join(f"{bmrb_seq[i]}{i + 1} (GT {gt_n[i]:.1f})" for i in order if gt_err[i] > TOP_THRESH) or "none")

# mean |error| with vs without the two largest errors (aliased T13 / S24)
drop = order[:2]
all_err = np.array(list(gt_err.values()))
kept_err = np.array([gt_err[i] for i in gt_err if i not in drop])
print("dropped:", ", ".join(f"{bmrb_seq[i]}{i + 1} {gt_err[i]:.1f} (GT {gt_n[i]:.1f})" for i in drop))
print(f"all     (n={len(all_err)}): mean {all_err.mean():.2f} ppm, median {np.median(all_err):.2f} ppm")
print(f"w/o top2 (n={len(kept_err)}): mean {kept_err.mean():.2f} ppm, median {np.median(kept_err):.2f} ppm")
print(f"difference: mean {all_err.mean() - kept_err.mean():.2f} ppm "
      f"({(all_err.mean() - kept_err.mean()) / all_err.mean():.0%} of the all-residue mean)")


## 2. Write the B-factor PDB and the .pml

In [ ]:
for i, res in enumerate(af_res):                               # residue i of the chain = BMRB index i (checked above)
    val = gt_err.get(i, SENTINEL)
    for atom in res:
        atom.set_bfactor(val)
out_pdb = f"{PID}_AF2_gt_error_bfactor.pdb"
io = PDBIO()
io.set_structure(af_struct)
io.save(os.path.join(OUT_DIR, out_pdb))
print("wrote", out_pdb)

obj = "ub_g76c_af2"
png = f"{PID}_AF2_gt_error.png"
# NOTE: no ';' anywhere in the .pml (even in comments) -- PyMOL treats it as a command separator
pml = f"""bg_color white
set ray_opaque_background, 0
set orthoscopic, on

set specular, 0
set reflect, 0
set ambient, 1
set depth_cue, 0
set ray_trace_mode, 1
set ray_trace_depth_factor, 1
set ray_trace_disco_factor, 1
set ray_trace_gain, 0.005
set ray_shadow, off
set antialias, 2
set cartoon_oval_length, 1
set cartoon_rect_length, 1

########################################
# {PID} -- ubiquitin G76C, AF2 model colored by ground-truth |error| (oof_resid, 15N, ppm)
# B-factor -1 = no data (grey)
########################################
load {out_pdb}, {obj}
hide everything
dss {obj}
show cartoon, {obj}

select {obj}_nodata, {obj} and b < -0.5
select {obj}_hasdata, {obj} and b > -0.5
color grey50, {obj}_nodata
spectrum b, white_red, {obj}_hasdata, minimum={VMIN:.1f}, maximum={VMAX:.1f}

select daf_top5, {obj} and b > {TOP_THRESH:g}
show sticks, daf_top5 and (sidechain or name CA+N)
show spheres, daf_top5 and name N
hide everything, hydro
set stick_radius, 0.35, daf_top5
set sphere_scale, 0.35, daf_top5 and name N

orient {obj}
deselect
#ray 1800, 1500
#png {png}, dpi=300
"""
assert ";" not in pml
pml_path = os.path.join(OUT_DIR, f"render_gt_error_{PID}_AF2.pml")
with open(pml_path, "w") as fh:
    fh.write(pml)
print("wrote", os.path.basename(pml_path))